# Feature Engineering

Questo notebook esplora le feature estratte durante la fase di preprocessing in formato tabellare.


In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ENV_PATH = PROJECT_ROOT / ".env"
ENV = dotenv_values(ENV_PATH)

def project_path(env_key, default):
    value = Path(ENV.get(env_key, default))
    return value if value.is_absolute() else PROJECT_ROOT / value

USING_SAMPLE = True

if not USING_SAMPLE:
    PROCESSED_PATH = project_path("PROCESSED_DATA_PATH", "data/processed/jmail_emails_processed.parquet")
    if not PROCESSED_PATH.exists():
        PROCESSED_PATH = PROJECT_ROOT / "data/processed/jmail_emails_processed_sample.parquet"
else:
    PROCESSED_PATH = PROJECT_ROOT / "data/processed/jmail_emails_processed_sample.parquet"

pd.set_option("display.max_columns", 80)


## 1. Caricamento Dati e Anteprima
Carichiamo i dati processati e visualizziamo la tabella con le feature generate.

In [2]:
processed = pd.read_parquet(PROCESSED_PATH)
print(f"Processed input: {PROCESSED_PATH}")
print(f"Righe: {len(processed):,}")
print(f"Colonne: {processed.shape[1]}")

feature_cols = [c for c in processed.columns if c not in ["combined_text", "sender_email", "recipient_emails"]]
# Mostriamo le prime righe del dataset con le nuove feature
processed[feature_cols].head(10)


Processed input: /home/filippo/Scrivania/pizza-cluster/data/processed/jmail_emails_processed_sample.parquet
Righe: 42,471
Colonne: 40


,id,doc_id,message_index,sender,subject,to_recipients,cc_recipients,bcc_recipients,sent_at,content_markdown,attachments,email_drop_id,is_promotional,release_batch,epstein_is_sender,all_participants,subject_clean,content_clean,subject_length,content_length,combined_text_length,has_subject,has_redaction,redaction_count,word_count,uppercase_ratio,sent_at_datetime,sent_year,sent_month,sent_dayofweek,sent_hour,is_weekend,has_sender,has_attachments,attachment_count,person_unknown,recipient_count_estimate,sender_domain,is_epstein_involved
0,000fa4d83ae293b30ed3571c9ea2fd77,905350832d97dc3675bfed84993cd746,0,jeffrey E. <jeevacation@gmail.com>,Re: Title and subtitle,"[""Ehud Barak <ehbarak1@gmail.com>""]",[],NaN,2015-05-07T07:07:33.000Z,"prefer OUR COUNTRY , MY LIFE or M...",0,ehud_ddos_dropsite_1,False,8,True,"jeffrey e. <jeevacation@gmail.com> [""ehud ba...",Re: Title and subtitle,"prefer OUR COUNTRY , MY LIFE or MOMENTS",22,39,63,True,False,0,12,0.396825,2015-05-07 07:07:33+00:00,2015,5,3,7,False,True,False,0,False,1,gmail.com,True
1,001612df62eb14194162f0a366793927,e7d35dec89fc8b6f9a988e766df31f0e,0,J. Epstein <jeeproject@yahoo.com>,Re: Barbro Ehnbom: SALSS 2006,"[""Cecilia Steen <cecilia.steen@gmail.com>""]",[],[],2006-07-10T18:13:50.000Z,i'll try\n\n--- Cecilia Steen <cecilia.steen@g...,0,yahoo_2,False,1,True,"j. epstein <jeeproject@yahoo.com> [""cecilia s...",Re: Barbro Ehnbom: SALSS 2006,i'll try --- Cecilia Steen <cecilia.steen@gmai...,29,684,715,True,False,0,125,0.058741,2006-07-10 18:13:50+00:00,2006,7,0,18,False,True,False,0,False,1,yahoo.com,True
2,001df92b110e9da90631a66cf97a0a11,cffd46d5af8f9bc29fb07179ab1ca5c3,0,J. Epstein <jeeproject@yahoo.com>,Re:,"[""<gmax1@mindspring.com>""]",[],[],2007-02-19T20:50:39.000Z,both\n\n----- Original Message ----\nFrom: Gma...,0,yahoo_2,False,1,True,"j. epstein <jeeproject@yahoo.com> [""<gmax1@mi...",Re:,both ----- Original Message ---- From: Gmax <g...,3,454,459,True,False,0,58,0.04793,2007-02-19 20:50:39+00:00,2007,2,0,20,False,True,False,0,False,1,yahoo.com,True
3,0036be5386133888b4201acacd0d6178,8d92ba5b22c17768b8cd2bd9ed8e7003,0,Ryan Andrews <ryan.andrews@sf-email.sharefile....,Infographic: 5 Ways to Take Center Stage at Work,"[""<jeeproject@yahoo.com>""]",[],[],2017-05-18T20:56:45.000Z,<http://pages.sharefile.com/dc/6-MALPEjw79s-xM...,0,yahoo_2,False,1,False,ryan andrews <ryan.andrews@sf-email.sharefile...,Infographic: 5 Ways to Take Center Stage at Work,<http://pages.sharefile.com/dc/6-MALPEjw79s-xM...,48,3866,3916,True,False,0,188,0.279111,2017-05-18 20:56:45+00:00,2017,5,3,20,False,True,False,0,False,1,sf-email.sharefile.com,False
4,0037a69505ae78dfacfebd97260f5325,182308de78c20502f23026e6c975d663,0,The New York Times <nytimes@e.newyorktimesinfo...,Don't Miss Our Biggest Sale! Save 50% for 26 W...,"[""<jeeproject@yahoo.com>""]",[],[],2014-11-26T19:03:47.000Z,Get 50% off 26 Weeks on a Digital or Home Deli...,0,yahoo_2,False,1,False,the new york times <nytimes@e.newyorktimesinf...,Don't Miss Our Biggest Sale! Save 50% for 26 W...,Get 50% off 26 Weeks on a Digital or Home Deli...,75,1575,1652,True,False,0,167,0.144673,2014-11-26 19:03:47+00:00,2014,11,2,19,False,True,False,0,False,1,e.newyorktimesinfo.com,False
5,003be449ea1eec7d6cc1b872e5190016,b260d23c23ce7c703c0a2196b7db9190,0,Lesley Groff <lesley@nysgllc.com>,FW: A message from Dr. Jarecki,"[""J. Epstein <jeeproject@yahoo.com>""]",[],[],2007-12-21T01:32:36.000Z,\n\n \n\nFrom: Michelle L. Acitelli [mailto:m...,1,yahoo_2,False,1,False,"lesley groff <lesley@nysgllc.com> [""j. epstei...",FW: A message from Dr. Jarecki,From: Michelle L. Acitelli [mailto:mla@falconf...,30,212,244,True,False,0,36,0.106557,2007-12-21 01:32:36+00:00,2007,12,4,1,False,True,True,1,False,1,nysgllc.com,True
6,004690537c9d8177b9bfd793416adc44,3b4a4b7496d2663b210643c8fc62ec4e,0,J. Epstein <jeeproject@yahoo.com>,Re:,"[""<gmax1@mindspring.com>""]",[],[],2006-11-28T04:37:03.000Z,yes cooridnate with Rich\n\n----- Original Mes...,0,yahoo_2,False,1,True,"j. epstein

## 2. Statistiche Descrittive delle Feature Numeriche
Di seguito le statistiche riassuntive (media, min, max, ecc.) per le nuove metriche calcolate.

In [3]:
processed.describe().round(2)

,message_index,attachments,release_batch,subject_length,content_length,combined_text_length,redaction_count,word_count,uppercase_ratio,sent_year,sent_month,sent_dayofweek,sent_hour,attachment_count,recipient_count_estimate
count,42471.00,42471.00,42471.00,42471.0,42471.0,42471.0,42471.0,42471.0,42471.0,41894.0,41894.0,41894.0,41894.0,42471.0,42471.0
mean,1.83,0.24,7.46,29.36,744.88,776.13,0.38,116.08,0.11,2015.44,6.41,2.43,14.26,0.24,2.38
std,3.68,41.30,4.02,26.35,3934.56,3935.75,1.36,248.59,0.1,3.17,3.58,1.68,7.75,41.3,2.65
min,0.00,0.00,1.00,0.0,0.0,3.0,0.0,1.0,0.0,1990.0,1.0,0.0,0.0,0.0,1.0
25%,0.00,0.00,1.00,12.0,129.0,160.0,0.0,26.0,0.05,2014.0,3.0,1.0,7.0,0.0,1.0
50%,0.00,0.00,10.00,23.0,326.0,361.0,0.0,57.0,0.09,2016.0,6.0,2.0,17.0,0.0,1.0
75%,2.00,0.00,10.00,40.0,738.0,772.0,0.0,120.0,0.13,2018.0,10.0,4.0,20.0,0.0,3.0
max,41.00,8511.00,12.00,240.0,360225.0,360363.0,103.0,9830.0,0.82,2035.0,12.0,6.0,23.0,8511.0,131.0


## 3. Analisi Tabellare: Mittenti e Censure
Raggruppiamo i dati per capire i domini mittente più attivi e quante email contengono censure o mostrano un certo numero di censure.

In [4]:
if 'sender_domain' in processed.columns:
    display(processed.groupby('sender_domain').size().reset_index(name='email_count').sort_values('email_count', ascending=False).head(15))

if 'has_redaction' in processed.columns:
    display(processed.groupby('has_redaction').size().reset_index(name='email_count'))

if 'redaction_count' in processed.columns:
    display(processed.groupby('redaction_count').size().reset_index(name='email_count').sort_values('redaction_count'))


,sender_domain,email_count
91,gmail.com,4151
218,yahoo.com,1769
45,db.com,327
158,nysgllc.com,324
15,aol.com,293
67,ellmax.com,197
159,nysgmail.com,95
63,e.newyorktimesinfo.com,93
65,earthlink.net,88
52,discard.mail.db.com,82


,has_redaction,email_count
0,False,35986
1,True,6485


,redaction_count,email_count
0,0,35986
1,1,2534
2,2,1285
3,3,1657
4,4,486
5,5,194
6,6,111
7,7,71
8,8,45
9,9,33


## 4. Analisi Tabellare: Distribuzione per Giorno della Settimana
Tabella con il numero di email inviate nel fine settimana rispetto ai giorni feriali.

In [5]:
if 'is_weekend' in processed.columns:
    weekend_table = processed.groupby('is_weekend').size().reset_index(name='email_count')
    weekend_table['Tipologia'] = weekend_table['is_weekend'].map({True: 'Weekend', False: 'Feriale'})
    display(weekend_table[['Tipologia', 'is_weekend', 'email_count']])


,Tipologia,is_weekend,email_count
0,Feriale,False,38041
1,Weekend,True,4430
